In [1]:
# Step 1: Import
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense
from sklearn.model_selection import train_test_split

In [2]:
# Step 2: Load & Clean Data
df = pd.read_csv("NER dataset.csv", encoding="latin1")
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [3]:
df["Sentence #"] = df["Sentence #"].ffill()
df["Word"] = df["Word"].bfill()

In [4]:
# Step 3: Create Sentences
sentences = df.groupby("Sentence #")["Word"].apply(list).values
tags = df.groupby("Sentence #")["POS"].apply(list).values

In [5]:
# Step 4: Tokenization
tok = Tokenizer()
tok.fit_on_texts(sentences)
X = tok.texts_to_sequences(sentences)

tag_tok = Tokenizer()
tag_tok.fit_on_texts(tags)
y = tag_tok.texts_to_sequences(tags)

In [6]:
# Step 5: Padding
MAX_LEN = 128
X = pad_sequences(X, maxlen=MAX_LEN, padding='post')
y = pad_sequences(y, maxlen=MAX_LEN, padding='post')

In [7]:
# Step 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [8]:
# Step 7: Model
model = Sequential([
    Embedding(len(tok.word_index)+1, 64),
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(np.max(y)+1, activation='softmax'))
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



In [9]:
# Step 8: Train
model.fit(X_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 70s 55ms/step - accuracy: 0.9546 - loss: 0.2016 
Epoch 2/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 71s 59ms/step - accuracy: 0.9927 - loss: 0.0255 
Epoch 3/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 73s 61ms/step - accuracy: 0.9946 - loss: 0.0166 
Epoch 4/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 68s 57ms/step - accuracy: 0.9953 - loss: 0.0134 
Epoch 5/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 66s 55ms/step - accuracy: 0.9959 - loss: 0.0115


In [10]:
# Step 9: Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9933 - loss: 0.0203
Accuracy: 0.9933416247367859


In [11]:
# Step 10: Prediction
sample = sentences[0]
seq = pad_sequences(tok.texts_to_sequences([sample]), maxlen=MAX_LEN, padding='post')

pred = np.argmax(model.predict(seq), axis=-1)[0]

idx2tag = {v:k for k,v in tag_tok.word_index.items()}

for w, t in zip(sample, pred[:len(sample)]):
    print(w, "->", idx2tag.get(t))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 958ms/step
Thousands -> nns
of -> in
demonstrators -> nns
have -> vbp
marched -> vbn
through -> in
London -> nnp
to -> to
protest -> vb
the -> dt
war -> nn
in -> in
Iraq -> nnp
and -> cc
demand -> nn
the -> dt
withdrawal -> nn
of -> in
British -> jj
troops -> nns
from -> in
that -> dt
country -> nn
. -> .
